In [1]:
import sys
import os 


sys.path.append('../../../DB')
import DB_utils

## 전체 테이블 정보를 dictionary로 가져온다.
conn = DB_utils.join_db()
results = DB_utils.ppt_info_search(conn, 1862)

In [2]:
import yaml
import sys
import os 


sys.path.append('../../../DB')
import DB_utils
with open("./configuration.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

In [3]:
def g(d, *keys, default=""):
    for k in keys:
        if not isinstance(d, dict):
            return default
        d = d.get(k)
        if d is None:
            return default
    return d

def f(d, *keys, default=""):
    for k in keys:
        # dict 접근
        if isinstance(d, dict):
            d = d.get(k, default)

        # list 접근
        elif isinstance(d, list) and isinstance(k, int):
            if 0 <= k < len(d):
                d = d[k]
            else:
                return default

        else:
            return default

        if d is None:
            return default

    return d


In [4]:
## 모두 값이 없을 수 있음

bd_name = g(results, "building_info", "bd_name")

middle_img1 = f(results, "images", "main", 0, "url")
middle_img2 = f(results, "images", "sub1", 0, "url")
middle_img3 = f(results, "images", "sub3", 0, "url")

gi1 = g(results, "building_info", "address")
gi2 = g(results, "building_info", "zoning_type")
gi3 = g(results, "building_info", "land_area_sqm")
gi3_1 = g(results, "building_info", "land_area_pyeong")

gi4 = g(results, "building_info", "gross_area_sqm")
gi4_1 = g(results, "building_info", "gross_area_pyeong")

gi5 = g(results, "building_info", "building_coverage_ratio")
gi6 = g(results, "building_info", "floor_area_ratio")

gi7 = g(results, "building_info", "aboveground_floors")
gi8 = g(results, "building_info", "underground_floors")

gi9 = g(results, "building_info", "building_usage")
gi10 = g(results, "building_info", "building_structure")

gi11 = g(results, "building_info", "parking_capacity")
gi12 = g(results, "building_info", "elevator")
gi13 = g(results, "building_info", "emergency_elevator")

gi14 = g(results, "building_info", "direction")
gi15 = g(results, "building_info", "approval_date")
gi16 = g(results, "building_info", "official_price_per_sqm_won")

feature_text = g(results, "building_memo", "bd_feature").split("\n")

right1 = g(results, "building_info", "price_per_pyeong")
right2 = g(results, "building_info", "price_per_total_floor_area")

right3 = g(results, "building_info", "security_deposit")
right4 = g(results, "building_info", "monthly_rent_fee")
right5 = g(results, "building_info", "maintenance_fee")



page2_img1 = f(results, "images", "sub5", 0, "url")
page2_img2 = f(results, "images", "sub5", 1, "url")
page2_img3 = f(results, "images", "sub5", 2, "url")
page2_img4 = f(results, "images", "sub5", 3, "url")
page2_img5 = f(results, "images", "sub5", 4, "url")
page2_img6 = f(results, "images", "sub5", 5, "url")

page4_img1 = f(results, "images", "sub3", 0, "url")
page4_img2 = f(results, "images", "sub4", 0, "url")

In [7]:
from pptx import Presentation
from pptx.util import Inches, Cm, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from datetime import datetime
import yaml
import sys
import os 


sys.path.append('../../../DB')
import DB_utils
with open("./configuration.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)


# 상단 이름 넣기 
def input_name_info(ppt,text : list):
    tf = ppt.text_frame 
    tf.clear()  # 기존 텍스트 제거
    lines = text
    
    for i, text in enumerate(lines):
        if i == 0:
            p = tf.paragraphs[0]
        else:
            p = tf.add_paragraph()

            # ⭐ 문단 간 여백 제거
        run = p.add_run()
        run.text = text # 택스트 적용 
        run.font.name = "NanumGothic"
        
        run.font.size = Pt(9)
        run.font.color.rgb = RGBColor(0, 0, 0)
        # 줄별로 각각 어떤 포인트로 할 지 설정해야함
        if i == 0:    
            run.font.color.rgb = RGBColor(11, 50, 121)  # 파란색
        elif i== 4:
            run.font.bold = False
            p.space_before = Pt(0)
            p.space_after = Pt(0)
            p.line_spacing = 1.0    
            run.font.size = Pt(8)

        else:
            run.font.bold = False
            p.space_before = Pt(0)
            p.space_after = Pt(0)
            p.line_spacing = 1.0
                    
        # 정렬 
        p.alignment = PP_ALIGN.LEFT

# 특정 공간에 이미지 넣기 공용 
def input_img(ppt,img_path):
    left = ppt.left
    top = ppt.top
    width = ppt.width
    height = ppt.height

    slide.shapes.add_picture(
        img_path,
        left,
        top,
        width=width,
        height=height
    )

    # 기존 도형 제거 (사진으로 교체 느낌)
    slide.shapes._spTree.remove(ppt._element)

GI_data = [
    ["대지위치",gi1],
    ["지역지구",gi2],
    ["대지면적",gi3 +"/"+gi3_1],
    ["연 면 적",gi4 +"/"+gi4_1],
    ["건폐율/용적률", gi5+"/"+gi6],
    ["건물규모",f"지상{gi7}층 / 지하 {gi8}층"],
    ["건축물주용도",gi9],
    ["건축물주구조",gi10],
    ["주차대수",gi11],
    ["승 강 기", f"승용 {gi12}대 / 비상 {gi13}대"],
    ["방향(주출입구기준)",gi14],
    ["사용승인일",gi15],
    ["공시지가",f"{gi16}원"]
]

# table 관련 
def make_GI(ppt, rows =13, cols = 2):
    left   = ppt.left
    top    = ppt.top
    width  = ppt.width
    height = ppt.height

    table = slide.shapes.add_table(
        rows, cols, left, top, width, height
    ).table

    table.first_row = False  # 머리글 행 특수 서식 해제
    table.first_col = False  # 첫 번째 열 특수 서식 해제
    # 너비 조절 (Cm 사용)
    table.columns[0].width = Cm(2.73)
    table.columns[1].width = Cm(5.32) # 전체 width에 맞춰 조절 필요

    gray = RGBColor(231, 230, 230)  # 연한 회색 (배경)
    white = RGBColor(255, 255, 255) # 흰색 (배경)
    black = RGBColor(0, 0, 0)       # 검정색 (글자)

    for r in range(rows):
        # 모든 행의 높이를 일정하게 맞추고 싶다면 아래 주석 해제
        #table.rows[r].height = Cm(6.81) 
        #table.rows[r].width = Cm(8.04) 

        for c in range(cols):
            cell = table.cell(r, c)
            cell.margin_top = Cm(0.0)
            cell.margin_bottom = Cm(0.0)
            # 배경색 설정
            cell.fill.solid()
            if c == 0:
                cell.fill.fore_color.rgb = gray
            else:
                cell.fill.fore_color.rgb = white

            cell.vertical_anchor = MSO_ANCHOR.MIDDLE
            # 폰트 설정 (폰트 8pt 적용)
            paragraph = cell.text_frame.paragraphs[0]
            if (c %2) == 0:
                paragraph.alignment = PP_ALIGN.CENTER  # 중앙 정렬 (필요시)
            else:
                paragraph.alignment = PP_ALIGN.LEFT
            # 텍스트가 이미 있거나 새로 넣을 때 폰트 적용을 위해 run 생성
            run = paragraph.add_run()
            run.font.name = "NanumGothic"
            run.font.size = Pt(8)
            run.font.color.rgb = black
            
            # 예시 텍스트 입력 (내용이 필요할 경우)
            run.text = GI_data[r][c]

# def Features 
def featrue_info(ppt, text):
    ppt.width = Cm(8.16)
    tf = ppt.text_frame
    tf.clear()  # 기존 서식 제거

    tf.vertical_anchor = MSO_ANCHOR.TOP
    lines = text  # 입력받은 텍스트 리스트

    for i, line_text in enumerate(lines):
        if i == 0:
            p = tf.paragraphs[0]
        else:
            p = tf.add_paragraph()

        # ⭐ 불렛 포인트(동그라미) 설정
        p.level = 0  # 불렛 수준 설정 (기본 0)
        
        # 문단 간 여백 및 줄 간격 제거 (촘촘하게)
        #p.space_before = Pt(1)
        p.line_spacing = 1.3
        
        # 정렬: 왼쪽 정렬
        p.alignment = PP_ALIGN.LEFT

        run = p.add_run()
        # 텍스트 앞에 동그라미 기호 추가 (가장 확실한 방법)
        run.text = f"• {line_text}" 
        
        # 폰트 기본 설정 (나눔고딕, 10pt)
        run.font.name = "NanumGothic"
        run.font.size = Pt(10)
        run.font.bold = False
        run.font.color.rgb = RGBColor(0, 0, 0) # 기본 검정색

# 텍스트 넣기 

color_define = {
    "red" : (205,35,50),
    "blue" : (11,50,121),
    "white" : (255,255,255)
}
def input_oneline_text(ppt,text,color = "red", fsize = 28 , bold = False ):
    tf = ppt.text_frame 
    tf.clear()  # 기존 텍스트 제거
    lines = text
    
    for i, text in enumerate(lines):
        if i == 0:
            p = tf.paragraphs[0]
        else:
            # 중 추가 
            p = tf.add_paragraph()

            # ⭐ 문단 간 여백 제거
        run = p.add_run()
        run.text = text # 택스트 적용 
        run.font.name = "NanumGothic"
        run.font.size = Pt(fsize)

        r, g, b = color_define[color]
        run.font.color.rgb = RGBColor(r,g,b)
        # 줄별로 각각 어떤 포인트로 할 지 설정해야함
       
        run.font.bold = bold              
        # 정렬 
        p.alignment = PP_ALIGN.RIGHT

def input_title(ppt, text):
    tf = ppt.text_frame
    tf.clear()                      # 기존 서식 제거

    p = tf.paragraphs[0]
    run = p.add_run()
    run.text = text

    run.font.name = "NanumGothic"   # 나눔고딕으로 변경 
    run.font.size = Pt(28)          # font 변경 
    run.font.bold = True            # ✅ 볼드




### 해당 정보를 바탕으로 ppt 만들기 진행 
conn = DB_utils.join_db()
results = DB_utils.ppt_info_search(conn, 53584)


# ppt load 
prs = Presentation("../statics/template.pptx")

# 대상 슬라이드 설정 
slide = prs.slides[0] # 대상 슬라이드 찾기 

# 지정된 이름의 객체 선택 
for shape in slide.shapes:
    print("shape:",shape)
    
    # title 설정 
    if shape.name == "title":
        input_title(shape,"Test")
        
    ## main / sub person name 넣기
    elif shape.name == "main_person":
        input_name_info(shape,cfg["main_person"])
    elif shape.name == "sub_person":
        input_name_info(shape, cfg["sub_person"])
    
    ## middle image 입력 
    elif shape.name == "middle_img1":
        if middle_img1 != "":
            input_img(shape, cfg["path"] + middle_img1)
    elif shape.name == "middle_img2":
        if middle_img2 != "":
            input_img(shape, cfg["path"] + middle_img2)
    elif shape.name == "middle_img3":
        if middle_img3 != "":
            input_img(shape, cfg["path"] + middle_img3)
    elif shape.name == "middle_img4":
        if middle_img4 != "":
            input_img(shape, cfg["path"] + middle_img4)
        
        
        
    # GI 입력
    elif shape.name == "GI_table":
        print("dddd")
        make_GI(shape)
        
    elif shape.name == "features":
        featrue_info(shape, feature_text)


    elif shape.name =="ask_price":
        input_oneline_text(shape, ["asking price"], color = 'red', fsize=28 , bold = True)

    elif shape.name =="area_price":
        input_oneline_text(shape, [right1,right2],color = 'blue', fsize= 10 )
    elif shape.name =="rent_price":
        input_oneline_text(shape, [right3,right4,right5],color = 'blue', fsize= 10)

    elif shape.name =="date":

        now = datetime.now()
        date_str = now.strftime("%Y년 %m월")
        input_oneline_text(shape, [date_str],color = 'white', fsize= 9 , bold= True)


##################### page 2  
slide = prs.slides[1] # 대상 슬라이드 찾기 

# 지정된 이름의 객체 선택 
for shape in slide.shapes:
    print("shape:",shape)
    
    ## main / sub person name 넣기
    if shape.name == "main_person":
        input_name_info(shape, cfg["main_person"])
    elif shape.name == "sub_person":
        input_name_info(shape, cfg["sub_person"])
    
    ## middle image 입력 
    elif (shape.name == "img1") and (page2_img1 != ""):
        input_img(shape, cfg["path"] + page2_img1)
    elif (shape.name == "img2") and (page2_img2 != ""):
        input_img(shape, cfg["path"] + page2_img2)
    elif (shape.name == "img3") and (page2_img3 != ""):
        input_img(shape, cfg["path"] + page2_img3)
    elif (shape.name == "img4") and (page2_img4 != ""):
        input_img(shape, cfg["path"] + page2_img4)
    elif (shape.name == "img5") and (page2_img5 != ""):
        input_img(shape, cfg["path"] + page2_img5)
    elif (shape.name == "img6") and (page2_img6 != ""):
        input_img(shape, cfg["path"] + page2_img6)

##################### page 3  
slide = prs.slides[2] # 대상 슬라이드 찾기 

# 지정된 이름의 객체 선택 
for shape in slide.shapes:
    print("shape:",shape)
    
    ## main / sub person name 넣기
    if shape.name == "main_person":
        input_name_info(shape, cfg["main_person"])
    elif shape.name == "sub_person":
        input_name_info(shape, cfg["sub_person"])
    
    ## middle image 입력 
    # elif shape.name == "img1":
    #     input_img(shape, "../img/1.png")


##################### page 4  
slide = prs.slides[3] # 대상 슬라이드 찾기 

# 지정된 이름의 객체 선택 
for shape in slide.shapes:
    print("shape:",shape)
    
    ## main / sub person name 넣기
    if shape.name == "main_person":
        input_name_info(shape, cfg["main_person"])
    elif shape.name == "sub_person":
        input_name_info(shape, cfg["sub_person"])
    
    ## middle image 입력 
    elif (shape.name == "img1") and (page4_img1 !=""):
        input_img(shape, cfg["path"] + page4_img1)
    elif (shape.name == "img2") and (page4_img2 !=""):
        input_img(shape, cfg["path"] + page4_img2)


prs.save("../statics/output.pptx")


shape: <pptx.shapes.graphfrm.GraphicFrame object at 0x00000255750A9510>
shape: <pptx.shapes.autoshape.Shape object at 0x0000025572ED6A10>
shape: <pptx.shapes.autoshape.Shape object at 0x0000025572E233D0>
shape: <pptx.shapes.graphfrm.GraphicFrame object at 0x00000255750F75D0>
shape: <pptx.shapes.autoshape.Shape object at 0x0000025574ACD690>
shape: <pptx.shapes.autoshape.Shape object at 0x00000255750F75D0>
shape: <pptx.shapes.autoshape.Shape object at 0x0000025572E13F10>
shape: <pptx.shapes.autoshape.Shape object at 0x00000255750A8410>
dddd
shape: <pptx.shapes.autoshape.Shape object at 0x00000255750F75D0>
shape: <pptx.shapes.autoshape.Shape object at 0x000002557242B290>
shape: <pptx.shapes.autoshape.Shape object at 0x00000255750A8410>
shape: <pptx.shapes.placeholder.SlidePlaceholder object at 0x00000255750F75D0>
shape: <pptx.shapes.autoshape.Shape object at 0x00000255750A8ED0>
shape: <pptx.shapes.autoshape.Shape object at 0x0000025571EF5CD0>
shape: <pptx.shapes.graphfrm.GraphicFrame obje

In [34]:
## table
from pptx import Presentation
from pptx.util import Inches, Cm, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR

prs = Presentation("./template.pptx")
slide = prs.slides[0]
rows = 13 
cols = 2

for shape in slide.shapes:
    if shape.name == "GI_img":
        left   = shape.left
        top    = shape.top
        width  = shape.width
        height = shape.height

        table = slide.shapes.add_table(
            rows, cols, left, top, width, height
        ).table

        table.first_row = False  # 머리글 행 특수 서식 해제
        table.first_col = False  # 첫 번째 열 특수 서식 해제
        # 너비 조절 (Cm 사용)
        table.columns[0].width = Cm(2.73)
        table.columns[1].width = Cm(5.32) # 전체 width에 맞춰 조절 필요

        gray = RGBColor(231, 230, 230)  # 연한 회색 (배경)
        white = RGBColor(255, 255, 255) # 흰색 (배경)
        black = RGBColor(0, 0, 0)       # 검정색 (글자)

        for r in range(rows):
            # 모든 행의 높이를 일정하게 맞추고 싶다면 아래 주석 해제
            #table.rows[r].height = Cm(6.81) 
            #table.rows[r].width = Cm(8.04) 

            for c in range(cols):
                cell = table.cell(r, c)
                cell.margin_top = Cm(0.0)
                cell.margin_bottom = Cm(0.0)
                # 배경색 설정
                cell.fill.solid()
                if c == 0:
                    cell.fill.fore_color.rgb = gray
                else:
                    cell.fill.fore_color.rgb = white

                cell.vertical_anchor = MSO_ANCHOR.MIDDLE
                # 폰트 설정 (폰트 8pt 적용)
                paragraph = cell.text_frame.paragraphs[0]
                paragraph.alignment = PP_ALIGN.CENTER  # 중앙 정렬 (필요시)
                
                # 텍스트가 이미 있거나 새로 넣을 때 폰트 적용을 위해 run 생성
                run = paragraph.add_run()
                run.font.name = "NanumGothic"
                run.font.size = Pt(8)
                run.font.color.rgb = black
                
                # 예시 텍스트 입력 (내용이 필요할 경우)
                run.text = f"Row {r}, Col {c}"

prs.save("output.pptx")

PackageNotFoundError: Package not found at './template.pptx'

# feature

In [37]:

from pptx.util import Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN

prs = Presentation("./template.pptx")
slide = prs.slides[0]


text = [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9"
]

for shape in slide.shapes:
    print("shape:",shape)
    if shape.name == "features":
        tf = shape.text_frame
        tf.clear()  # 기존 서식 제거


        tf.vertical_anchor = MSO_ANCHOR.TOP
        lines = text  # 입력받은 텍스트 리스트

        for i, line_text in enumerate(lines):
            if i == 0:
                p = tf.paragraphs[0]
            else:
                p = tf.add_paragraph()

            # ⭐ 불렛 포인트(동그라미) 설정
            p.level = 0  # 불렛 수준 설정 (기본 0)
            
            # 문단 간 여백 및 줄 간격 제거 (촘촘하게)
            p.space_before = Pt(0)
            p.space_after = Pt(0)
            p.line_spacing = 1.0
            
            # 정렬: 왼쪽 정렬
            p.alignment = PP_ALIGN.LEFT

            run = p.add_run()
            # 텍스트 앞에 동그라미 기호 추가 (가장 확실한 방법)
            run.text = f"• {line_text}" 
            
            # 폰트 기본 설정 (나눔고딕, 10pt)
            run.font.name = "NanumGothic"
            run.font.size = Pt(10)
            run.font.bold = False
            run.font.color.rgb = RGBColor(0, 0, 0) # 기본 검정색

prs.save("output.pptx")

shape: <pptx.shapes.autoshape.Shape object at 0x000001E7ECA70C50>
shape: <pptx.shapes.graphfrm.GraphicFrame object at 0x000001E7ECA73610>
shape: <pptx.shapes.graphfrm.GraphicFrame object at 0x000001E7E68213D0>
shape: <pptx.shapes.autoshape.Shape object at 0x000001E7ECA70750>
shape: <pptx.shapes.autoshape.Shape object at 0x000001E7ECA71C90>
shape: <pptx.shapes.autoshape.Shape object at 0x000001E7ECA70C50>
shape: <pptx.shapes.autoshape.Shape object at 0x000001E7ECA73D90>
shape: <pptx.shapes.autoshape.Shape object at 0x000001E7ECA71C90>
shape: <pptx.shapes.placeholder.SlidePlaceholder object at 0x000001E7ECA700D0>
shape: <pptx.shapes.autoshape.Shape object at 0x000001E7DF585610>
shape: <pptx.shapes.autoshape.Shape object at 0x000001E7ECA71C90>


In [ ]:
def input_oneline_text(ppt,text,color = "red", fsize = 15):
    tf = ppt.text_frame 
    tf.clear()  # 기존 텍스트 제거
    lines = text
    

    p = tf.paragraphs[0]
    p = tf.add_paragraph()

    # ⭐ 문단 간 여백 제거
    run = p.add_run()
    run.text = text # 택스트 적용 
    run.font.name = "NanumGothic"
        
    run.font.size = Pt(fsize)
    run.font.color.rgb = RGBColor(0, 0, 0)
    run.font.bold = False
    p.space_before = Pt(0)
    p.space_after = Pt(0)
    p.line_spacing = 1.0    
    run.font.size = Pt(8)                    
    # 정렬 
    p.alignment = PP_ALIGN.LEFT
